In [ ]:
!pip install mediapipe==0.10.11 opencv-python numpy matplotlib pillow

In [ ]:
!pip install body-matrix

In [ ]:
!pip uninstall mediapipe -y
!pip install mediapipe==0.10.11

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import math

# Preprocessing & Landmark Detection

In [ ]:
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

pose = mp_pose.Pose(static_image_mode=True, model_complexity=1, enable_segmentation=False, min_detection_confidence=0.5)

In [ ]:
class CameraCalibrator:
  def __init__(self):
    self.focal_length = None
    self.calibrated = False

  def calibrate_camera(self, known_distance_cm, pixel_width):

    """
    Calibrate camera using: F = (Pknown x Dknown)
    """
    self.focal_length = (pixel_width * known_distance_cm)
    self.calibrated = True
    return self.focal_length

  def calculate_distance(self, real_width_cm, pixel_width):

    """
    Calculate actual distance: D = (W x F) / P
    """
    if not self.calibrated:
      raise ValueError("Camera is not calibrated")
    return (real_width_cm * self.focal_length) / pixel_width

In [ ]:
class PoseValidator:
  def __init__(self, min_distance = 150):
    self.min_distance = min_distance
    self.required_landmarks = [
      mp_pose.PoseLandmark.NOSE,
      mp_pose.PoseLandmark.LEFT_SHOULDER,
      mp_pose.PoseLandmark.RIGHT_SHOULDER,
      mp_pose.PoseLandmark.LEFT_ELBOW,
      mp_pose.PoseLandmark.RIGHT_ELBOW,
      mp_pose.PoseLandmark.LEFT_WRIST,
      mp_pose.PoseLandmark.RIGHT_WRIST,
      mp_pose.PoseLandmark.LEFT_HIP,
      mp_pose.PoseLandmark.RIGHT_HIP,
      mp_pose.PoseLandmark.LEFT_KNEE,
      mp_pose.PoseLandmark.RIGHT_KNEE,
      mp_pose.PoseLandmark.LEFT_ANKLE,
      mp_pose.PoseLandmark.RIGHT_ANKLE
    ]

  def validate_landmarks(self, landmarks):
    """Check if all required landmarks are detected with visibility > 0.5"""
    if not landmarks:
      return False, "No landmarks detected"

    missing_landmarks = []
    for landmark_id in self.required_landmarks:
      landmark = landmarks.landmark[landmark_id]
      if landmark.visibility < 0.5:
        missing_landmarks.append(mp_pose.PoseLandmark(landmark_id).name)

    if missing_landmarks:
      return False, f"Missing landmarks: {missing_landmarks}"

    return True, "All landmarks detected"

  def validate_vertical_pose(self, landmarks):
    """Validate standing pose: Ynose < Yshoulders < Yhips < Yknees < Yankles"""

    nose_y = landmarks.landmark[mp_pose.PoseLandmark.NOSE].y

    shoulder_y = (landmarks.landmark[mp_pose.PoseLandmark.LEFT_SHOULDER].y + landmarks.landmark[mp_pose.PoseLandmark.RIGHT_SHOULDER].y)/2

    hip_y = (landmarks.landmark[mp_pose.PoseLandmark.LEFT_HIP].y + landmarks.landmark[mp_pose.PoseLandmark.RIGHT_HIP].y)/2

    knee_y = (landmarks.landmark[mp_pose.PoseLandmark.LEFT_KNEE].y + landmarks.landmark[mp_pose.PoseLandmark.RIGHT_KNEE].y)/2

    ankle_y = (landmarks.landmark[mp_pose.PoseLandmark.LEFT_ANKLE].y + landmarks.landmark[mp_pose.PoseLandmark.RIGHT_ANKLE].y)/2

    if nose_y < shoulder_y < hip_y < knee_y < ankle_y:
      return True, "Correct standing pose"

    return False, "Incorrect pose - please stand upright"

  def validate_distance(self, calibrator, landmarks, image_width):
    """Check if user is at correct distance (≥1.5m)"""
    # Calculate shoulder width in pixels
    left_shoulder = landmarks.landmark[mp_pose.PoseLandmark.LEFT_SHOULDER]
    right_shoulder = landmarks.landmark[mp_pose.PoseLandmark.RIGHT_SHOULDER]
    shoulder_pixel_width = abs(left_shoulder.x - right_shoulder.x) + image_width

    # Estimate distance using average shoulder width (40cm)
    estimated_distance = calibrator.calculate_distance(40, shoulder_pixel_width)

    if estimated_distance >= self.min_distance:
      return True, f"Distance OK: {estimated_distance:.2f} cm"
    return False, f"Too close: {estimated_distance:.2f} cm (minimum: {self.min_distance})"


In [ ]:
class BodyLandmarkDetector:
    def __init__(self):
        self.pose = pose
        self.landmarks = None

    def detect_landmarks(self, image):
        """Detect body landmarks using MediaPipe"""
        rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        results = self.pose.process(rgb_image)

        if results.pose_landmarks:
            self.landmarks = results.pose_landmarks
            return True, results.pose_landmarks
        return False, None

    def extract_keypoints(self, landmarks, image_shape):
        """Extract normalized coordinates for key body points"""
        h, w = image_shape[:2]
        keypoints = {}

        landmark_mapping = {
            'nose': mp_pose.PoseLandmark.NOSE,
            'left_shoulder': mp_pose.PoseLandmark.LEFT_SHOULDER,
            'right_shoulder': mp_pose.PoseLandmark.RIGHT_SHOULDER,
            'left_elbow': mp_pose.PoseLandmark.LEFT_ELBOW,
            'right_elbow': mp_pose.PoseLandmark.RIGHT_ELBOW,
            'left_wrist': mp_pose.PoseLandmark.LEFT_WRIST,
            'right_wrist': mp_pose.PoseLandmark.RIGHT_WRIST,
            'left_hip': mp_pose.PoseLandmark.LEFT_HIP,
            'right_hip': mp_pose.PoseLandmark.RIGHT_HIP,
            'left_knee': mp_pose.PoseLandmark.LEFT_KNEE,
            'right_knee': mp_pose.PoseLandmark.RIGHT_KNEE,
            'left_ankle': mp_pose.PoseLandmark.LEFT_ANKLE,
            'right_ankle': mp_pose.PoseLandmark.RIGHT_ANKLE
        }

        for name, landmark_id in landmark_mapping.items():
            landmark = landmarks.landmark[landmark_id]
            keypoints[name] = {
                'x': landmark.x * w,
                'y': landmark.y * h,
                'visibility': landmark.visibility
            }

        return keypoints


# Measurement Calculation

In [ ]:
class BodyMeasurementCalculator:
    def __init__(self, scale_factor=0.185):
        self.scale_factor = scale_factor

    def euclidean_distance(self, point1, point2):
        """Calculate Euclidean distance between two points"""
        return math.sqrt((point1['x'] - point2['x'])**2 + (point1['y'] - point2['y'])**2)

    def calculate_linear_measurements(self, keypoints):
        """Calculate linear body measurements"""
        measurements = {}

        # Shoulder width
        measurements['shoulder_width'] = self.euclidean_distance(
            keypoints['left_shoulder'], keypoints['right_shoulder']
        ) * self.scale_factor

        # Torso length (neck to hip)
        neck_point = {
            'x': (keypoints['left_shoulder']['x'] + keypoints['right_shoulder']['x']) / 2,
            'y': (keypoints['left_shoulder']['y'] + keypoints['right_shoulder']['y']) / 2
        }
        hip_center = {
            'x': (keypoints['left_hip']['x'] + keypoints['right_hip']['x']) / 2,
            'y': (keypoints['left_hip']['y'] + keypoints['right_hip']['y']) / 2
        }
        measurements['torso_length'] = self.euclidean_distance(
            neck_point, hip_center
        ) * self.scale_factor

        # Leg length (hip to ankle)
        measurements['leg_length'] = self.euclidean_distance(
            keypoints['left_hip'], keypoints['left_ankle']
        ) * self.scale_factor

        # Arm length (shoulder to wrist approximation)
        measurements['arm_length'] = self.euclidean_distance(
            keypoints['left_shoulder'], keypoints['left_wrist']
        ) * self.scale_factor

        # Hip breadth
        measurements['hip_breadth'] = self.euclidean_distance(
            keypoints['left_hip'], keypoints['right_hip']
        ) * self.scale_factor

        # Total height (nose to ankle)
        measurements['total_height'] = self.euclidean_distance(
            keypoints['nose'], keypoints['left_ankle']
        ) * self.scale_factor

        return measurements

    def calculate_circular_measurements(self, linear_measurements):
        """Calculate circular measurements using anthropometric ratios"""
        circular = {}

        # Chest circumference (approximately 3.2 * shoulder width)
        circular['chest_circumference'] = linear_measurements['shoulder_width'] * 2.8

        # Waist circumference (approximately 2.8 * hip breadth)
        circular['waist_circumference'] = linear_measurements['hip_breadth'] * 3.4

        # Hip circumference (approximately 3.1 * hip breadth)
        circular['hip_circumference'] = linear_measurements['hip_breadth'] * 3.6

        return circular


# Main Application

In [ ]:
class BodyMeasurementApp:
  def __init__(self):
    self.calibrator = CameraCalibrator()
    self.validator = PoseValidator()
    self.detector = BodyLandmarkDetector()
    self.calculator = BodyMeasurementCalculator()

  def capture_image(self):
    from google.colab import files
    uploaded = files.upload()

    for filename in uploaded.keys():
      image = cv2.imread(filename)
      return self.process_image(image)

  def process_image(self, image):
    """Main processing pipeline"""
    # Step 1: Detect landmarks
    success, landmarks = self.detector.detect_landmarks(image)
    if not success:
      return {"error": "No landmarks detected. Ensure good lighting and clear background."}

    # Step 2: Validate landmarks
    valid_landmarks, landmark_msg = self.validator.validate_landmarks(landmarks)
    if not valid_landmarks:
      return {"error": landmark_msg}

    # Step 3: Validate pose
    valid_pose, pose_msg = self.validator.validate_vertical_pose(landmarks)
    if not valid_pose:
      return {"error": pose_msg}

    # Step 4: Validate distance (requires calibration)
    if self.calibrator.calibrated:
      valid_distance, distance_msg = self.validator.validate_distance(self.calibrator, landmarks, image.shape[1])
      if not valid_distance:
        return {"error": distance_msg}

    # Step 5: Extract keypoints and calculate measurements
    keypoints = self.detector.extract_keypoints(landmarks, image.shape)
    linear_measurements = self.calculator.calculate_linear_measurements(keypoints)
    circular_measurements = self.calculator.calculate_circular_measurements(linear_measurements)

    return {
        "linear_measurements": linear_measurements,
        "circular_measurements": circular_measurements,
        "keypoints": keypoints
    }

  def display_results(self, results):
    if "error" in results:
      print(f"❌ Error: {results['error']}")
      return

    print("✅ Body Measurements Successfully Calculated!")
    print("\n📏 LINEAR MEASUREMENTS (cm):")
    for measurement, value in results['linear_measurements'].items():
      print(f"  • {measurement.replace('_', ' ').title()}: {value:.1f} cm")

    print("\n⭕ CIRCULAR MEASUREMENTS (cm):")
    for measurement, value in results['circular_measurements'].items():
      print(f"  • {measurement.replace('_', ' ').title()}: {value:.1f} cm")


In [ ]:
# Initialize the application
app = BodyMeasurementApp()


#Using my shoulder width (44 cm) standing at 150 cm away from the lens
app.calibrator.calibrate_camera(known_distance_cm = 150, pixel_width=233)

# Process image with proper error handling
try:
    results = app.capture_image()
    if results:  # Make sure results is not None
        app.display_results(results)
    else:
        print("No results returned from image processing")
except Exception as e:
    print(f"Error processing image: {e}")
